# 03 — Modelo de ML AML

Modelo PF em nível cliente-mês usando label fraco baseado em regras AML.


## Lógica

1. Agregar transações por cliente-mês.
2. Juntar KYC, comportamento geográfico e semelhança por profissão.
3. Criar label fraco: `suspicious_label = 1` quando `rule_count >= 3`.
4. Fazer split temporal: meses antigos para treino, meses recentes para validação.
5. Treinar XGBoost com `random_state=42`.
6. Avaliar AUC-PR, AUC-ROC, precision, recall, FPR e MCC por threshold.


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path('../data/raw/AML Case Cloudwalk INC (2).xlsx')
xls = pd.ExcelFile(DATA)
transactions = pd.read_excel(xls, 'Transactions')
kyc = pd.read_excel(xls, 'KYC_Profiles')
merchants = pd.read_excel(xls, 'Merchants')
geo = pd.read_excel(xls, 'GeoBehavior')

transactions.shape, kyc.shape, merchants.shape, geo.shape


In [ ]:
import sys
sys.path.append('../src')
from features import build_customer_month_dataset

dataset, rule_cols = build_customer_month_dataset(transactions, kyc, merchants, geo)
dataset[['customer_id','month','tx_count','total_amount','rule_count','suspicious_label']].head()


In [ ]:
dataset.groupby(['month','suspicious_label']).size().unstack(fill_value=0)


In [ ]:
from ml_model import fit_xgboost_pf, temporal_split, get_feature_columns
from sklearn.metrics import average_precision_score, roc_auc_score

pf = dataset[dataset['entity_type_model'].eq('PF')].copy()
pipe = fit_xgboost_pf(dataset, rule_cols)
train, valid = temporal_split(pf)
cat_cols, num_cols = get_feature_columns(pf, rule_cols)
for col in cat_cols:
    valid[col] = valid[col].fillna('__MISSING__').astype(str)
X_valid = valid[cat_cols + num_cols]
y_valid = valid['suspicious_label'].astype(int)
prob = pipe.predict_proba(X_valid)[:, 1]
average_precision_score(y_valid, prob), roc_auc_score(y_valid, prob)


## Observação importante

Este modelo não substitui análise humana nem comunicação formal. Ele prioriza a fila com base em padrões aprendidos a partir de regras explicáveis. Em produção, seria necessário calibrar threshold, revisar falsos positivos e validar estabilidade temporal.
